In [ ]:
import pandas as pd
import duckdb
import os
import requests
from tqdm import tqdm 
import time
from concurrent.futures import ThreadPoolExecutor
import numpy as np

### BRONZE LAYER

In [ ]:
## Here, I am backfilling current seasons data as I started this project around GW 23ish and storinng it in a seperate table 
# which will be merged later onto the historical in the silver layer. And after a few transofrmation, a master table will be available in the Gold layer.
from ingestion_scripts.weekly_sync import *
run_fpl_pipeline()

#### The reason why I am ingesting data only from 2016-17 to 2024-25:
    - I want to add the column/feature 'code' (an unique id) for all the players from Chris Musson's FPL_id_map because vaastav has not added code in his dataset (in the merged gws)
    - And clean the data to sync with my weekly ingest and current season backfill maps the code from the bootstrapstatic while ingesting, so we don't need to worry about that later

In [ ]:
# Ingesting and cleaning historical data
filepath = os.path.join(os.getcwd(), "data/bronze/FPL_historical.parquet")

In [ ]:

fpl_seasons = ["2016-17", "2017-18", "2018-19", "2019-20", "2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

base_url = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/"

def fetch_fpl_season(season):
    url = f"{base_url}{season}/gws/merged_gw.csv"
    try:
        df = pd.read_csv(url, encoding='latin1', low_memory=False)
        df['season'] = season
        exclude_cols = ["id", "GW", "completed_passes", "dribbles", "ea_index", 
                        "errors_leading_to_goal", "errors_leading_to_goal_attempt", 
                        "fouls", "key_passes", "offside", "open_play_crosses", 
                        "big_chances_created", "big_chances_missed", "mng_win", 
                        "mng_loss", "mng_draw", "mng_clean_sheets", "mng_goals_scored", 
                        "mng_underdog_draw", "mng_underdog_win", "kickoff_time_formatted", 
                        "loaned_in", "loaned_out", "penalties_conceded", "tackled", 
                        "target_missed", "winning_goals"]
        
        df.drop(columns=exclude_cols, inplace=True, errors='ignore')
        if 'position' in df.columns:
            df = df[df['position'] != 'AM']
            
        df['kickoff_time'] = pd.to_datetime(df['kickoff_time']).dt.strftime('%Y-%m-%d %H:%M:%S')
        print(f"Successfully downloaded: {season}")
        return df
    
    except Exception as e:
        print(f"Error ingesting {season}: {e}")
        return None

print(f"Starting parallel download for {len(fpl_seasons)} seasons...")

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_fpl_season, fpl_seasons))

all_seasons_data = [df for df in results if df is not None]

if all_seasons_data:
    final_df = pd.concat(all_seasons_data, axis=0, ignore_index=True)
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    final_df.to_parquet(filepath, index=False, compression='snappy')
    
    print(f"\n Hitorical data ingest, complete!")
    print(f"Total rows: {len(final_df)}")
else:
    print("No data was recovered. Check your URLs or internet connection.")

In [ ]:
print(len(pd.read_parquet("data/bronze/FPL_historical.parquet")))

In [ ]:
## Code mapping historical data as its a unique id and the current weekly ingest takes this into account
# Getting the fpl_id map from github
id_filepath = r"C:\Programing\Projects\Fantasy Premier League\data\bronze\FPL_idmap.parquet"
elementxcode_url = "https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/main/FPL/"
seasons = ["16-17", "17-18", "18-19", "19-20", "20-21", "21-22", "22-23", "23-24", "24-25"]

def fetch_season(season):
    url = f"{elementxcode_url}{season}.csv"
    try:
        df = pd.read_csv(url)
        if season in df.columns:
            df = df.rename(columns={season: 'element'})
        df['season'] = f"20{season}"
        return df
    except Exception as e:
        print(f"Could not retrieve {season}: {e}")
        return None

with ThreadPoolExecutor(max_workers=5) as executor:
    id_results = list(executor.map(fetch_season, seasons))

id_data = pd.concat([df for df in id_results if df is not None], ignore_index=True).to_parquet(id_filepath)

duckdb.execute(f"""
    CREATE OR REPLACE TABLE memory_join AS 
    SELECT 
        df2.code::VARCHAR AS code, 
        df1.*
    FROM read_parquet('{filepath}') AS df1
    LEFT JOIN read_parquet('{id_filepath}') AS df2
        ON df1.element = df2.element
        AND df1.season = df2.season
""")

duckdb.execute(f"COPY memory_join TO '{filepath}' (FORMAT 'PARQUET')")

duckdb.execute("DROP TABLE memory_join")

print(f"Successfully joined 'code' and updated: FPL_historical")


### SILVER LAYER

#### Imputing nulls and position/team mapping 
    Once imputed and pos/team mapped always concatenate `data/silver/FPL_cleaned` with the weekly updatyed current fpl season

In [ ]:
test_path = "data/bronze/FPL_historical copy.parquet"
ustat_path = "data/bronze/understat_historical.parquet"
fpl_path = "data/bronze/FPL_historical.parquet"
umatch = "data/bronze/EPL_match_summary.parquet"
temp_path= "temp.parquet"
curr = "data/bronze/current_fpl_season.parquet"

In [ ]:
## Columns to be analyzed 
df = pd.read_parquet(fpl_path)
df.columns[df.isnull().any()].tolist()   

In [ ]:

## Combine understat id to test_fpl
from master_map import bankai
bankai(test_path, ustat_path = None, output_path=test_path)

In [ ]:
## There is this one GHOST game which was postponed but still in the dataset fixture - 27, ARS vs MAN CITY
duckdb.sql(f"""
    COPY (
        SELECT * FROM read_parquet('{test_path}')
        WHERE NOT (fixture = 275 AND season = '2019-20' AND kickoff_time = '2020-03-11 19:30:00')
    ) TO '{test_path}' (FORMAT PARQUET);
""")

In [ ]:
## Creating the ultimate ustat for season 2106/17 - 2024/25
duckdb.sql(f""" 
    SELECT player_id, m.match_id, position as u_position, home_team, away_team, h_goals, a_goals, h_xg, a_xg, m.datetime, m.season
    FROM read_parquet('{ustat_path}') as u
    LEFT JOIN read_parquet('{umatch}') as m
    ON u.match_id = m.match_id
""").write_parquet(temp_path)

In [ ]:
## Imputing/Cleaning historical FPL
duckdb.sql(f""" 
    WITH score_map AS (
        SELECT 
            DISTINCT 
            temp.match_id,
            test.fixture,
            temp.away_team, 
            temp.home_team,
            temp.a_goals,
            temp.h_goals,
            temp.h_xg,
            temp.a_xg,
            test.kickoff_time,
            temp.season
        FROM (SELECT *, CAST(kickoff_time AS TIMESTAMP) as ts_fpl FROM read_parquet('{test_path}')) as test
        LEFT JOIN (SELECT *, CAST(datetime AS TIMESTAMP) as ts_u FROM read_parquet('{temp_path}')) as temp
        ON CAST(test.u_id AS VARCHAR) = CAST(temp.player_id AS VARCHAR)
        AND CAST(test.season AS VARCHAR) = CAST(temp.season AS VARCHAR)
        AND test.ts_fpl BETWEEN (temp.ts_u - INTERVAL 12 HOURS) AND (temp.ts_u + INTERVAL 12 HOURS)
        WHERE match_id is not null
        QUALIFY ROW_NUMBER() OVER(PARTITION BY test.u_id, test.ts_fpl ORDER BY abs(date_diff('minute', test.ts_fpl, temp.ts_u)) ASC) = 1  
    )
        SELECT 
            f.* exclude (u_id)
            replace(
            coalesce(team, case when was_home = true then home_team else away_team end) as team,
            coalesce(team_a_score, a_goals) as team_a_score, 
            coalesce(team_h_score, h_goals) as team_h_score,
            CASE 
                WHEN was_home = true then away_team else home_team
                END AS opponent_team
            )
        FROM read_parquet('{test_path}') as f
        LEFT JOIN score_map as s
        ON f.fixture = s.fixture
        AND f.season = s.season
""").write_parquet("data/silver/FPL_cleaned")
   # code, name, position, team, was_home, home_team, away_team, team_a_score, team_h_score, expected_goals_conceded, a_goals, h_goals, a_xg, h_xg

In [ ]:
duckdb.sql(""" 
        SELECT season, count(distinct fixture)
    FROM read_parquet("data/silver/FPL_cleaned")
    GROUP BY season 
    """)

In [ ]:
## Imputing null positions using understat and mapping player position
from master_map import shikai
shikai(fpl_path="data/silver/FPL_cleaned", output_path="data/silver/FPL_cleaned", ustat_path= ustat_path)

## Added the u_position in bankai to fill the positions alone

In [ ]:
duckdb.sql(""" 
        SELECT * EXCLUDE(u_position)
           REPLACE(
           COALESCE(position, u_position) as position
           )
        FROM read_parquet("data/silver/FPL_cleaned")
    """).write_parquet("data/silver/FPL_cleaned")

In [ ]:
from master_map import patch_work
patch_work("data/silver/FPL_cleaned")

#### Feature Engineering

In [ ]:
df = pd.read_parquet("data/silver/FPL_cleaned")
#display(df.info())
df.columns[df.isnull().any()].tolist()   

In [ ]:
fpl_cleaned = "data/silver/fpl_cleaned"
current_fpl = "data/bronze/current_fpl_season.parquet"
fpl_merged = "data/silver/FPL_merged"
current_fixture = "data/bronze/FPL_fixture.parquet"
fplxuid = "data/silver/FPL_uid"

In [ ]:
## The merge
duckdb.sql(f"""
        SELECT * FROM read_parquet('{fpl_cleaned}')
        UNION BY NAME
        SELECT * FROM read_parquet('{current_fpl}')
""").write_parquet(fpl_merged)

In [ ]:
# xGc calculation
## Delete the fplxuid everytime you run it
from master_map import bankai
bankai(fpl_merged, fplxuid, None)

duckdb.sql(f""" 
    WITH ultimate_ustat AS(
        SELECT 
            player_id, 
            m.match_id, 
            position as u_position, 
            u.team_name,
            home_team, away_team, 
            h_goals, a_goals, 
            h_a,
            h_xg, 
            a_xg, 
            CASE 
                WHEN h_a = 'h' then a_xg
                ELSE h_xg
                END as xGc,
            m.datetime, 
            m.season
        FROM read_parquet('{ustat_path}') as u
        LEFT JOIN read_parquet('{umatch}') as m
        ON u.match_id = m.match_id
    ),
    score_map AS (
        SELECT 
            DISTINCT 
            temp.match_id,
            test.fixture,
            temp.team_name,
            temp.xGc,
            test.kickoff_time,
            temp.season
        FROM (SELECT *, CAST(kickoff_time AS TIMESTAMP) as ts_fpl FROM read_parquet('{fplxuid}')) as test
        LEFT JOIN (SELECT *, CAST(datetime AS TIMESTAMP) as ts_u FROM ultimate_ustat) as temp
        ON CAST(test.u_id AS VARCHAR) = CAST(temp.player_id AS VARCHAR)
        AND CAST(test.season AS VARCHAR) = CAST(temp.season AS VARCHAR)
        AND test.ts_fpl BETWEEN (temp.ts_u - INTERVAL 12 HOURS) AND (temp.ts_u + INTERVAL 12 HOURS)
        WHERE match_id is not null
        QUALIFY ROW_NUMBER() OVER(PARTITION BY test.u_id, test.ts_fpl ORDER BY abs(date_diff('minute', test.ts_fpl, temp.ts_u)) ASC) = 1  
        )
        Select * from score_map where season = '2025-26' order by kickoff_time, season, fixture 
""")

In [ ]:
duckdb.sql(f""" 
    WITH ultimate_ustat AS(
        SELECT 
            player_id, 
            m.match_id, 
            position as u_position, 
            u.team_name,
            home_team, away_team, 
            h_goals, a_goals, 
            m.datetime, 
            m.season
        FROM read_parquet('{ustat_path}') as u
        LEFT JOIN read_parquet('{umatch}') as m
        ON u.match_id = m.match_id
    ),
    score_map AS (
        SELECT 
            DISTINCT 
            temp.match_id,
            test.fixture,
            test.round,
            temp.home_team,
            temp.away_team,
            temp.h_goals,
            temp.a_goals,
            test.kickoff_time,
            temp.season
        FROM (SELECT *, CAST(kickoff_time AS TIMESTAMP) as ts_fpl FROM read_parquet('{test_path}')) as test
        LEFT JOIN (SELECT *, CAST(datetime AS TIMESTAMP) as ts_u FROM ultimate_ustat) as temp
        ON CAST(test.u_id AS VARCHAR) = CAST(temp.player_id AS VARCHAR)
        AND CAST(test.season AS VARCHAR) = CAST(temp.season AS VARCHAR)
        AND test.ts_fpl BETWEEN (temp.ts_u - INTERVAL 12 HOURS) AND (temp.ts_u + INTERVAL 12 HOURS)
        WHERE match_id is not null
        QUALIFY ROW_NUMBER() OVER(PARTITION BY test.u_id, test.ts_fpl ORDER BY abs(date_diff('minute', test.ts_fpl, temp.ts_u)) ASC) = 1  
        ),
    hist_bps AS (
        SELECT 
           fixture, 
           round, 
           sum(bps) filter (where was_home = true) as h_bps,
           sum(bps) filter (where was_home = false) as a_bps,
           kickoff_time, 
           season
        FROM read_parquet('{test_path}')
        GROUP BY kickoff_time, round, season, fixture
        order by season, round, kickoff_time
    )
        Select h.fixture as fixture_id, h.round, home_team, away_team, h_bps, a_bps, h_goals as h_score, a_goals as a_score, h.kickoff_time, h.season
        from hist_bps as h
        left join score_map as s
        on s.fixture = h.fixture
        and s.season = h.season
        order by h.kickoff_time, h.season, h.fixture 
""").write_parquet("data/bronze/FPL_historical_fixture.parquet")


In [ ]:
duckdb.sql(f"""
        SELECT * FROM read_parquet('{current_fixture}')
""")

In [ ]:
duckdb.sql("""
        SELECT * FROM read_parquet("data/bronze/FPL_historical_fixture.parquet")
        UNION BY NAME
        SELECT * FROM read_parquet("data/bronze/FPL_fixture.parquet")
""").write_parquet("data/silver/Fixture_merged")